In [2]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv, find_dotenv 
import os 

load_dotenv(find_dotenv(),override=True)

if os.environ["GROQ_API_KEY"]:
    print("Api key successfully found")
else:
    print("API Key not Found")

Api key successfully found


# **GraphSchema**

In [3]:
from typing import TypedDict,Optional,Literal
class graph_schema(TypedDict):
    task:str
    status:Optional[Literal["pending","approved","rejected"]]

# **Nodes**

In [6]:
from langgraph.types import Command,interrupt
def ask_approval(state:graph_schema)->Command:
    
    """
    This tool will pause the execution and asks user for his approval,
    and resumes exection from the paused state
    """
    response=interrupt({
        "messages":"Do you approve this task?",
        "task":state['task']
    })
    if response:
        return Command(goto="approve_task")
    else :
        return Command(goto="reject_task")

# Status functions 

In [7]:
def approve_task(state:graph_schema):
    print("Task approved")
    return {"status":"approved"}

def reject_task(state:graph_schema):
    print("Task Rejected")
    return {"status":"rejected"}

# **Build Graph**

In [8]:
from langgraph.graph import StateGraph ,START, END
from langgraph.checkpoint.memory import MemorySaver

graph=StateGraph(state_schema=graph_schema)

graph.add_node("ask",ask_approval)
graph.add_node("approve_task",approve_task)
graph.add_node("reject_task",reject_task)

graph.add_edge(START,"ask")
graph.add_edge("approve_task",END)
graph.add_edge("reject_task",END)

human_graph=graph.compile()